# Measuring Gendered Interruptions at the Supreme Court of Canada
## A Computational Analysis of Judicial Power Dynamics in Oral Argument

**Group Members:**
- Sabrin Saide (221178603)
- Matt Aydin (220185328)
- Gobind Dhugee (221173794)

**Course:** Legal Tech Coding — Professor Rehaag, Winter 2026

---

**Research Question:** Can AI-generated transcripts be used to measure whether gendered interruption patterns exist during Supreme Court of Canada oral hearings?


## 1. Setup

First, we clone our project repository from GitHub and install the required Python packages. This only needs to run once.

In [ ]:
import os
import subprocess
import sys

# Clone the repo if we haven't already
if not os.path.exists("scc-interruptions"):
    subprocess.run(["git", "clone", "https://github.com/matt-aydin-2000/scc-interruptions.git"], check=True)
    print("Repository cloned.")
else:
    print("Repository already exists.")

# Install dependencies
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "scc-interruptions/requirements.txt"], check=True)
print("Dependencies installed.")


## 2. Import Our Modules

We add the project directory to Python's path so we can import our custom modules.

In [ ]:
import sys
sys.path.insert(0, "scc-interruptions")
os.chdir("scc-interruptions")

from scraper import fetch_all_transcripts
from transcript_parser import parse_transcript_file
from interruption_detector import detect_interruptions, compute_interruption_metrics, compute_time_to_first_interruption
from analysis import build_justice_dataframe, build_case_level_dataframe, full_analysis, format_results_report
from visualizations import generate_all_visualizations

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("Agg")
%matplotlib inline

from IPython.display import Image, display

print("All modules imported successfully.")

## 3. Data Acquisition

We scrape 121 AI-generated SCC hearing transcripts from [obiter.ai](https://obiter.ai/scc/), Simon Wallace's project. Wallace transcribed these hearings using OpenAI's Whisper (speech-to-text) and Pyannote (speaker diarization). See: Wallace, S. (2023), "Speaking Like a Judge," *Supreme Court Law Review*, 115(1).

The scraper caches downloaded files, so re-running this cell won't re-download transcripts you already have.

In [ ]:
RAW_DATA_DIR = "data/raw"
PROCESSED_DIR = "data/processed"
OUTPUT_DIR = "output"

os.makedirs(RAW_DATA_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

fetch_all_transcripts(RAW_DATA_DIR, delay=1.0)


## 4. Parse Transcripts into Speaker Turns

Each transcript is HTML with a pattern like `Justice Kasirer (00:09:02): Vous n'allez pas...`. Our parser extracts the speaker name, timestamp, and text from each turn, then tags whether the speaker is a justice or counsel. Justice gender is coded from official SCC pronouns, following Wallace's methodology.

In [ ]:
raw_files = sorted([f for f in os.listdir(RAW_DATA_DIR) if f.endswith(".json")])

cases = []
for filename in raw_files:
    filepath = os.path.join(RAW_DATA_DIR, filename)
    try:
        case = parse_transcript_file(filepath)
        if case["turns"]:
            cases.append(case)
    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")

total_turns = sum(c["total_turns"] for c in cases)
print(f"Parsed {len(cases)} cases with {total_turns} total speaker turns.")

# Show a sample turn so you can see the data structure
print("\nSample speaker turn:")
for k, v in cases[0]["turns"][0].items():
    print(f"  {k}: {v}")


## 5. Detect Interruptions

We detect interruptions using three methods adapted from the US Supreme Court literature (Jacobi & Schweers 2017, Feldman & Gill 2019):

1. **Overlap** — the diarization model explicitly flagged overlapping speech
2. **Timing-based** — a new speaker starts within 15 seconds AND the previous speaker's text looks cut off (ends with a dash, comma, conjunction, etc.)
3. **Rapid judicial intervention** — a justice speaks before counsel has said more than 50 words

We use a 15-second threshold, calibrated through pilot testing.

In [ ]:
all_interruptions = []
all_metrics = []
all_case_data = []
all_ttfi = []

for case in cases:
    turns = case["turns"]
    interruptions = detect_interruptions(turns, time_threshold=15)
    all_interruptions.extend(interruptions)

    for intr in interruptions:
        intr["case_id"] = case["case_id"]
        intr["case_date"] = case["date"]

    metrics = compute_interruption_metrics(interruptions, turns)
    all_metrics.append((case["case_id"], metrics))
    all_case_data.append((case["case_id"], case["date"], metrics, case))

    ttfi = compute_time_to_first_interruption(turns, interruptions)
    all_ttfi.extend(ttfi)

n_overlap = sum(1 for i in all_interruptions if i["type"] == "overlap")
n_timing = sum(1 for i in all_interruptions if i["type"] == "timing")
n_rapid = sum(1 for i in all_interruptions if i["type"] == "rapid_intervention")

print(f"Total interruptions found: {len(all_interruptions)}")
print(f"  Overlapping speech:  {n_overlap}")
print(f"  Timing-based:        {n_timing}")
print(f"  Rapid intervention:  {n_rapid}")


## 6. Justice-Level Summary

We aggregate interruption counts across all cases for each justice, normalized per 1,000 words spoken to control for volubility (Feldman & Gill 2019 found that speaking an additional 1,000 words increases interruption rate by ~40%).

In [ ]:
justice_df = build_justice_dataframe(all_metrics)
case_df = build_case_level_dataframe(all_case_data)

display_cols = ["justice", "gender", "cases_heard", "total_words_spoken",
                "interruptions_made", "interruptions_received",
                "rate_made_per_1k_words", "rate_received_per_1k_words"]
justice_df[display_cols].round(2)


## 7. Statistical Analysis

We run the full battery of tests:
- **Descriptive statistics** (means, medians by gender)
- **Welch's t-tests** and **Mann-Whitney U tests** (comparing male vs female rates)
- **Negative binomial regression** (predicting interruption count from gender, controlling for volubility and Chief Justice status)
- **Cohen's d** effect sizes
- **Z-score outlier analysis**

In [ ]:
results = full_analysis(justice_df, case_df)
report = format_results_report(results)
print(report)


## 7b. Gendered Word-Count Dynamics

Professor Rehaag raised an important question: are there gendered dynamics in the number of words spoken? Our main analysis normalizes interruptions per 1,000 words to control for volubility (following Feldman & Gill 2019), but this choice has consequences. If male justices systematically speak more than female justices, normalization could mask or distort gender differences in raw interruption counts. We investigate this below.

In [ ]:
from analysis import gendered_word_count_analysis, format_word_count_report
from visualizations import plot_gendered_word_count

wc_results = gendered_word_count_analysis(justice_df, case_df)
print(format_word_count_report(wc_results))

# Generate the two-panel visualization
plot_gendered_word_count(justice_df, OUTPUT_DIR)
display(Image(filename=os.path.join(OUTPUT_DIR, "gendered_word_count_analysis.png")))

### Implications of Gendered Volubility

The analysis above shows whether male and female justices differ in how much they speak. This matters for two reasons:

1. **If male justices speak more**, then normalizing per 1,000 words could *deflate* their interruption rates relative to female justices. In that case, the absence of a normalized gender difference might mask a raw-count difference where men interrupt more in absolute terms.

2. **If word counts are similar**, then normalization has little effect and our main finding (no gender difference) is robust regardless of whether we normalize.

Either way, our regression models already control for volubility by including log(words_spoken) as a covariate. The regression coefficient for gender remains non-significant in both models, confirming that the null gender finding holds whether we control for speaking volume through normalization, through regression, or both. The choice to normalize is methodologically sound following Feldman & Gill (2019), but the result does not depend on it.

## 8. Visualizations

### Interruption Rates by Gender

In [ ]:
fig_paths = generate_all_visualizations(justice_df, all_interruptions, all_ttfi, OUTPUT_DIR)

for path in fig_paths:
    display(Image(filename=path))

## 9. Key Findings

**Interruptions made:** No statistically significant gender difference. Male justices averaged 2.58 interruptions per 1,000 words spoken; female justices averaged 2.51 (t-test p = 0.92, Mann-Whitney p = 0.48). Cohen's d = 0.057 (negligible effect size).

**Interruptions received:** Male justices appeared to receive more interruptions on average (2.39 vs 1.01 per 1,000 words), but this was driven almost entirely by Chief Justice Wagner (z-score = 2.98), whose procedural role as hearing manager inflates his count. After regression controls, gender was not a significant predictor (p = 0.97).

**Regression:** Neither model (interruptions made or received) found gender to be a significant predictor after controlling for volubility and Chief Justice status.

**Outlier:** Chief Justice Wagner is a strong outlier in both interruptions made and received, consistent with the unique procedural role of the Chief Justice in managing oral hearings.


## 10. COVID-19 Hearing Mode Analysis

The 2020–2022 period covered by our dataset spans a major shift in how SCC hearings were conducted:

| Period | Mode | Details |
|--------|------|---------|
| Jan 2020 – Mar 13, 2020 | **In-person** | Traditional courtroom hearings |
| Mar 14 – Jun 7, 2020 | **Suspended** | No hearings held |
| Jun 8, 2020 – Sep 12, 2021 | **Remote** | Zoom-based hearings |
| Sep 13, 2021 – onward | **Hybrid** | Justices in courtroom, counsel remote |

We classify each hearing by date and compare interruption patterns across modes. This addresses Professor Rehaag's feedback about noting which hearings were conducted remotely.

In [ ]:
from covid_analysis import add_hearing_mode_to_case_df, covid_summary

covid_report, case_df_with_mode = covid_summary(case_df)
print(covid_report)

### COVID Analysis Findings

Interruption rates declined significantly as hearings moved online:

- **In-person** hearings (8 cases) had the highest interruption rate: 2.23 per justice per case
- **Remote/Zoom** hearings (55 cases) dropped to 1.53 (p = 0.019 vs in-person)
- **Hybrid** hearings (58 cases) dropped further to 1.25 (p = 0.036 vs remote)

This suggests that the shift to remote hearings meaningfully reduced interruptions, likely because the turn-taking norms of video conferencing discourage overlapping speech. The absence of a significant gender difference persisted across all three hearing modes, consistent with our main findings.

We retain all hearing modes in our main analysis but note that the COVID-era shift may attenuate observed interruption rates compared to pre-pandemic norms.

## 11. Validation

Our professor's feedback highlighted validation as the main area for improvement: *"take a look at a random sample of sections of transcripts, manually review for interruptions, then report on accuracy of your automated process."*

We implement two forms of validation:

### A. Manual Validation (Precision & Recall Sampling)

To assess how well our automated pipeline performs, we conduct a stratified random sampling exercise with two components:

**Precision check (Part A):** We randomly select 30 interruptions that our algorithm *did* flag, stratified across all three detection methods (overlap, timing-based, and rapid judicial intervention). For each flagged interruption, we display the surrounding 5 turns of transcript context — enough to see whether the flagged speaker transition genuinely looks like an interruption or is a normal turn change. A human reviewer reads the context and judges: *Was the previous speaker actually cut off, or did they finish their thought naturally?* The precision rate is the proportion of flagged interruptions that a human reviewer agrees are real.

**Recall check (Part B):** We randomly select 30 speaker transitions that our algorithm did *not* flag as interruptions. We display the same 5-turn context window and ask: *Did a real interruption occur here that our algorithm missed?* The recall rate is the proportion of real interruptions that our algorithm successfully captured.

The validation materials are saved to `output/validation_sample.txt` with full context for each sample, allowing independent review. We display a preview of the first 3 precision samples below so the reader can see the format and judge the quality of our detection.

In [ ]:
from validation import save_validation_materials

val_report, precision_items, recall_items = save_validation_materials(
    cases, all_interruptions, OUTPUT_DIR, sample_size=30
)

# Show the first 3 precision samples as a preview
print("PREVIEW: First 3 items from the precision check\n")
for i, item in enumerate(precision_items[:3]):
    intr = item["interruption"]
    print(f"--- Sample {i+1} ---")
    print(f"Case: {item['case_id']}")
    print(f"Detection method: {intr['type']}")
    print(f"Interrupter: {intr['interrupter']} ({intr.get('interrupter_gender', '?')})")
    print(f"Interruptee: {intr['interruptee']} ({intr.get('interruptee_gender', '?')})")
    print("Context:")
    for turn in item["context_turns"]:
        marker = " >>>" if turn["turn_index"] == intr["turn_index"] else "    "
        text_preview = turn["text"][:100] + "..." if len(turn["text"]) > 100 else turn["text"]
        print(f"{marker} [{turn['timestamp_str']}] {turn['speaker']}: {text_preview}")
    print()

print(f"Full validation report saved to: output/validation_sample.txt")
print(f"(Contains {len(precision_items)} precision items + {len(recall_items)} recall items)")

### B. LLM Validation (GPT-4o-mini)

We also built a module (`llm_validator.py`) that sends each flagged interruption's transcript context to OpenAI's GPT-4o-mini and asks the model to:
1. Confirm or reject the interruption classification
2. Classify the type: **hostile**, **clarifying**, **procedural**, or **not an interruption**
3. Provide a brief explanation

#### Defining Interruption Types

We instructed the LLM to classify confirmed interruptions into three categories, drawing on the linguistic literature (Zimmerman & West 1975, Tannen 1994):

- **Hostile**: A power-asserting interruption where the interrupter cuts off the speaker mid-thought to challenge, dismiss, or redirect them. The interrupted speaker was clearly not finished, and the interruption conveys dominance or impatience. Example: a justice cuts off counsel mid-sentence with "That's not what I asked you" or "Move on to your next point."

- **Clarifying**: A substantive judicial intervention where the justice interjects to ask a genuine question, seek elaboration, or test a legal proposition. The interruption serves the purpose of understanding the argument better, even though it may technically cut off the speaker. Example: "But doesn't that conflict with the holding in *Carter*?"

- **Procedural**: A hearing-management interruption where the interrupter (typically the Chief Justice) intervenes to manage time, direct the order of speakers, or handle administrative matters. Example: "Counsel, your time has expired" or "We'll hear from the intervener now."

This typology matters because not all interruptions carry the same implications for gender dynamics. The US literature (Jacobi & Schweers 2017) focused primarily on hostile interruptions as markers of power imbalance. If most SCC interruptions are clarifying or procedural rather than hostile, the access-to-justice implications differ substantially.

**Note on cost limitations:** The LLM validation module requires an OpenAI API key, which is passed at runtime (never stored in the code) via `python llm_validator.py --api-key YOUR_KEY --sample 50`. We kept this as a separate manual step rather than integrating it into the main pipeline so that users maintain full control over API costs (~$0.50–2.00 for the full dataset). Below we display the results from a representative sample run.

In [ ]:
# Display the LLM validation results
llm_report_path = os.path.join(OUTPUT_DIR, "llm_validation_report.txt")
if os.path.exists(llm_report_path):
    with open(llm_report_path) as f:
        print(f.read())
else:
    print("LLM validation report not found. Run llm_validator.py to generate it.")
    print("Usage: python llm_validator.py --api-key YOUR_KEY --sample 50")

### C. How LLM Validation Affects Our Analysis

The LLM validation provides two important insights that refine our main findings:

**1. Precision estimate (~68%).** Our heuristic detection has a roughly 68% precision rate — meaning about one-third of flagged interruptions are actually normal speaker transitions. This is expected given the limitations of AI-generated transcripts, which lack the explicit interruption markers (double-dashes) available in US SCOTUS transcripts. Overlap detection performs best (83% confirmed), while timing-based detection has the most false positives (65% confirmed). This suggests that future iterations should weight overlap-detected interruptions more heavily.

**2. Interruption type classification.** Of the confirmed interruptions, 47% were **clarifying** (the justice asked a substantive question), 44% were **procedural** (hearing management, especially by Chief Justice Wagner), and only 9% were **hostile** (power-asserting cut-offs). This type distribution is important because it suggests that the vast majority of judicial interruptions at the SCC serve legitimate purposes. The low rate of hostile interruptions — and its similar distribution across male and female justices — further supports our main conclusion that gendered power dynamics do not clearly manifest in SCC interruption patterns.

**3. Gender-neutral confirmation rates.** The LLM confirmed interruptions at similar rates for male justices (62%) and female justices (70%), indicating that our heuristic methods do not systematically over- or under-count interruptions by gender. The main finding — no significant gender difference — is robust to LLM-based validation.

## 12. Limitations

- **Interruption detection is approximate.** We use timing heuristics rather than explicit markers (like the double-dash in US SCOTUS transcripts). Our validation sample (Section 11) allows us to quantify this imprecision.
- **Small sample of justices.** Only 10 justices served during this period (6 male, 4 female), limiting statistical power.
- **Chief Justice effect.** Wagner's procedural role distorts the "interruptions received" metric. We control for this in regression.
- **COVID-era remote hearings.** As shown in Section 10, interruption rates dropped significantly when hearings moved online (p = 0.019). We retain all modes in the main analysis but note this as a confound.
- **AI transcription errors and systematic bias.** Wallace's Whisper-based system makes errors that are not randomly distributed. Transcription accuracy likely degrades during overlapping speech segments — precisely the moments we are trying to detect as interruptions. This creates a systematic concern: the moments most important to our analysis are also the moments where the transcript is least reliable. Additionally, Whisper's word error rate may differ between English and French speech segments, and rapid code-switching (common at the SCC) can confuse the ASR model, leading to garbled or misattributed text. Diarization errors (assigning speech to the wrong speaker) compound this problem: if Speaker A's words are attributed to Speaker B during an overlap, our algorithm may either fabricate a nonexistent interruption or miss a real one. These errors could inflate or deflate our counts in ways that are difficult to quantify without a gold-standard human transcript for comparison.
- **Bilingual proceedings.** The SCC operates in both English and French, with justices and counsel frequently switching between languages within a single hearing — and sometimes within a single turn. This bilingual environment introduces challenges that our methodology does not fully address. French and English have different prosodic patterns (rhythm, stress, intonation), which may affect how Pyannote segments speaker turns. Interpretation segments, where a simultaneous interpreter translates for the bench, can create apparent "overlaps" that are not genuine interruptions. We did not separate interpreted segments from direct speech, which may introduce false positives in our overlap detection method. Future work should include language-specific validation and consider excluding interpreted segments from the interruption analysis.
- **No ground-truth benchmark.** Unlike the US SCOTUS (which has professional transcripts with interruption markers), there is no gold-standard Canadian transcript to validate against.

## 13. Future Directions: Video-Based Analysis

A promising avenue for future research would be to bypass text transcripts entirely and analyze the SCC's video recordings directly using multimodal large language models. The SCC publishes full video recordings of oral hearings on its website, and modern LLMs (such as GPT-4o, Gemini 1.5 Pro, or Claude) can now process video and audio inputs natively.

**Why this could improve accuracy:** Our current pipeline achieves approximately 68% precision because it relies on AI-generated *text* transcripts that have already lost important audio information — tone of voice, speech overlap timing, audible hesitations, and the acoustic signature of one speaker cutting off another. A video-based approach could:

1. **Detect overlapping speech directly from the audio waveform**, rather than relying on Pyannote's diarization output, which sometimes misses or fabricates overlaps.
2. **Use visual cues** — a justice leaning forward, a counsel pausing mid-gesture, or a visible reaction to being cut off — to disambiguate genuine interruptions from normal turn-taking.
3. **Leverage speaker identification from video** to resolve the diarization errors that currently plague our pipeline (e.g., misattributed speech during simultaneous talking).

**Practical considerations:** Processing 121 full hearing videos through a multimodal LLM would be computationally expensive (potentially $50–200+ in API costs at current pricing). However, a targeted approach — feeding only the 2-minute windows around each flagged interruption — could validate our existing results at a fraction of that cost. This would effectively use the video as a higher-fidelity ground truth against which to benchmark our text-based heuristics, potentially pushing precision well above the current 68%.

**Broader implications:** If video-based interruption detection proves more accurate, it could be applied to other courts that publish hearing recordings, including provincial appellate courts and the Federal Court of Appeal — none of which have publicly available transcripts. This would extend the reach of computational judicial behavior research far beyond courts that happen to have text transcripts available.

## 16. Save Data

In [ ]:
import json

# Save justice-level metrics
justice_df.to_csv(os.path.join(OUTPUT_DIR, "justice_metrics.csv"), index=False)

# Save case-level data (with hearing mode column)
case_df_with_mode.to_csv(os.path.join(OUTPUT_DIR, "case_level_metrics.csv"), index=False)

# Save interruptions data
with open(os.path.join(PROCESSED_DIR, "all_interruptions.json"), "w") as f:
    json.dump(all_interruptions, f, indent=2, ensure_ascii=False, default=str)

# Save time-to-first-interruption
if all_ttfi:
    pd.DataFrame(all_ttfi).to_csv(os.path.join(OUTPUT_DIR, "time_to_first_interruption.csv"), index=False)

# Save reports
with open(os.path.join(OUTPUT_DIR, "analysis_report.txt"), "w") as f:
    f.write(report)

with open(os.path.join(OUTPUT_DIR, "covid_analysis_report.txt"), "w") as f:
    f.write(covid_report)

print("All output saved to the output/ folder.")

## 15. AI Disclosure

We used AI tools at several stages of this project:

**Code development.** We used Claude (Anthropic) as a coding assistant throughout development. The Python modules (`scraper.py`, `transcript_parser.py`, `interruption_detector.py`, `analysis.py`, `visualizations.py`, `covid_analysis.py`, `validation.py`, `llm_validator.py`) were written collaboratively with Claude — we described what each module needed to do and iteratively refined the code it produced. We reviewed all code for correctness and made manual edits where needed. The research design decisions (choice of three detection methods, 15-second threshold, normalization approach, statistical tests) were informed by our reading of the US literature (Jacobi & Schweers 2017, Feldman & Gill 2019) and then implemented with AI assistance.

**LLM-based validation.** As described in Section 11B, we used OpenAI's GPT-4o-mini to classify detected interruptions by type (hostile, clarifying, procedural) and to provide an independent precision estimate for our heuristic detection. This is a substantive methodological component of the project, not just a development aid.

**Writing and analysis.** The markdown narrative in this notebook was drafted with AI assistance and then reviewed and edited by the group. The interpretive framing — particularly the access-to-justice implications, the discussion of bilingual proceedings as a limitation, and the future directions section on video-based multimodal analysis — reflects our own analytical choices, with AI helping to articulate them clearly.

**What was not AI-generated.** The research question, the decision to study the SCC specifically, the choice to use Wallace's obiter.ai transcripts, and the overall project structure were developed by the group. The data itself comes entirely from obiter.ai. All statistical results are computed by our code, not generated by an LLM.